In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, glob, re, warnings, numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.sans-serif':['Arial', 'DejaVu Sans'],
                     'axes.unicode_minus':False})

WV_B  = rf'{DATA_ROOT}/new_climate/wave'
WN_B  = rf'{DATA_ROOT}/new_climate/criso_wind'
E5_WV = rf'{DATA_ROOT}/AIS_processing/weather_data/wave'
E5_WN = rf'{DATA_ROOT}/AIS_processing/weather_data/wind'
M2C   = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid2'
OUT   = rf'{DATA_ROOT}/new_climate/_proj_main'
os.makedirs(OUT, exist_ok=True)

REF      = (2002, 2014)
M2_COVER = (2011, 2021)
BLOCKS   = [(2015,2024),(2025,2034),(2035,2044),(2045,2054),(2055,2064),
            (2065,2074),(2075,2084),(2085,2094),(2095,2100)]
SLICES   = {'2021-2040':(2021,2040),'2041-2060':(2041,2060),
            '2061-2080':(2061,2080),'2081-2100':(2081,2100)}
NQ       = 100
HS_CUTS  = [2.5, 4.0, 6.0, 9.0]
WD_CUTS  = [10.8, 13.9, 17.2, 20.8]
HS_EDGES = [0, 1.25, 2.5, 4.0, 6.0, 9.0, np.inf]
WD_EDGES = [0, 8.0, 10.8, 13.9, 17.2, 20.8, np.inf]
MIN_BASE = 0.5                      # baseline <0.5 d/yr → no percentage reported

MODELS = {
 'ACCESS': {'wave':{'his':rf'{WV_B}\ACCESS\CDFAC_his','ssp126':rf'{WV_B}\ACCESS\CDFAC1_126',
                    'ssp585':rf'{WV_B}\ACCESS\CDFAC1_585'},
            'wind':{'his':rf'{WN_B}\ACCESS-CM2\his','ssp126':rf'{WN_B}\ACCESS-CM2\126',
                    'ssp585':rf'{WN_B}\ACCESS-CM2\585'}},
 'ECEARTH':{'wave':{'his':rf'{WV_B}\ec_earth\cdfac1\his','ssp126':rf'{WV_B}\ec_earth\cdfac1\126',
                    'ssp585':rf'{WV_B}\ec_earth\cdfac1\585'},
            'wind':{'his':rf'{WN_B}\EC-EARTH3\his','ssp126':rf'{WN_B}\EC-EARTH3\126',
                    'ssp585':rf'{WN_B}\EC-EARTH3\585'}},
}
MDL_LABEL={'ACCESS':'ACCESS-CM2','ECEARTH':'EC-Earth3'}
SSP_LABEL={'ssp126':'SSP1-2.6','ssp585':'SSP5-8.5'}

REGIONS = {
 'NW_Europe':{'Iberia_Atlantic':[(36,45,-15,-6)],'Biscay_Celtic':[(45,63,-15,-6),(43,63,-6,0)],
              'North_Sea':[(48,63,0,13)]},
 'NW_Atlantic':{'GoM_Caribbean':[(18,31,-98,-60)],'NE_Coast':[(31,42,-82,-60),(42,50,-67,-60)],
                'Newfoundland':[(45,55,-60,-46)]},
 'Asia':{'SE_Asia':[(-10,20,95,130)],'E_Asia':[(20,46,117,146)]},
 'Med_Black':{'Med_West':[(30,43,-6,0),(30,46,0,19)],'Med_East':[(30,46,19,26.5),(30,41,26.5,37)],
              'Black':[(41,47.5,26.5,42)]},
 'Baltic':{'Baltic':[(53,66,13,30)]},
 'Indian_Ocean_N':{'Arabian':[(5,30,50,78)],'BoB':[(5,26,78,95)]},
}
FB=[(r,s,b) for r,su in REGIONS.items() for s,bs in su.items() for b in bs]
def assign_rs(la,lo):
    for r,s,(S,N,W,E) in FB:
        if S<=la<N and W<=lo<E: return r,s
    return None,None

CLAT=np.arange(-89,90,2.0); CLON=np.arange(-179,180,2.0)
GID=np.array([f'{a:.3f}_{b:.3f}' for a in CLAT for b in CLON])
rows=[]
for f in glob.glob(os.path.join(M2C,'*__dmax_wave.parquet')):
    for g in pd.read_parquet(f,columns=['grid_id']).grid_id.unique():
        la,lo=map(float,g.split('_')); r,s=assign_rs(la,lo)
        if s: rows.append((g,la,lo,r,s))
sea=(pd.DataFrame(rows,columns=['grid_id','clat','clon','region','subregion'])
     .drop_duplicates('grid_id').sort_values('grid_id').reset_index(drop=True))
pos={g:i for i,g in enumerate(GID)}; sea['flat_idx']=sea.grid_id.map(pos)
assert sea.flat_idx.notna().all(), 'grid_id does not match global cell grid'
SEL=sea.flat_idx.values.astype(int); NCELL=len(sea)
REGS=sorted(sea.region.unique())

LON_ORDER_05  = np.argsort(((np.arange(0,360,0.5)+180)%360)-180)
LON_ORDER_025 = np.argsort(((np.arange(0,360,0.25)+180)%360)-180)

def cell_05(dm):        # (nd,361,720) lat ascending → (nd,NCELL), max over 4×4=16 points
    d=dm[:,:360,:][:,:,LON_ORDER_05]
    c=np.nanmax(d.reshape(d.shape[0],90,4,180,4),axis=(2,4))
    return c.reshape(d.shape[0],-1)[:,SEL]

def cell_025(dm):       # (nd,721,1440) lat ascending → (nd,NCELL), max over 8×8=64 points
    d=dm[:,:720,:][:,:,LON_ORDER_025]
    c=np.nanmax(d.reshape(d.shape[0],90,8,180,8),axis=(2,4))
    return c.reshape(d.shape[0],-1)[:,SEL]

def win(m,half=1): return [((m-1+k)%12)+1 for k in range(-half,half+1)]

print(f'cells {NCELL} | subregions {sea.subregion.nunique()} | regions {len(REGS)}')
print(sea.groupby('region').size().to_string())


In [ ]:
def read_model(path, kind):
    ds=xr.open_dataset(path, drop_variables=['MAPSTA'])
    t=pd.to_datetime(ds['time'].values)
    if kind=='wave':
        f=ds['hs'].values.astype(np.float32)
    else:
        u=ds['uwnd'].values.astype(np.float32); v=ds['vwnd'].values.astype(np.float32)
        f=np.sqrt(u*u+v*v); del u,v                       # ★ compute wind speed at each 3-h step first
    ds.close()
    assert f.shape[0]%8==0 and t[0].hour==0, path
    nd=f.shape[0]//8
    return cell_05(np.nanmax(f.reshape(nd,8,361,720),axis=1)), t[::8].normalize()

def build_model(model, kind, tag):
    fo=os.path.join(OUT,f'cm_{kind}_{model}_{tag}.nc')
    if os.path.exists(fo):
        print(f'[CACHE] {os.path.basename(fo)}'); return xr.open_dataarray(fo).load()
    suf={'wave':'hs','wind':'wnd'}[kind]
    files=sorted(glob.glob(os.path.join(MODELS[model][kind][tag],f'ww3.*_{suf}.nc')))
    print(f'{model} {kind} {tag}: {len(files)} files',flush=True)
    A,T=[],[]
    for k,f in enumerate(files):
        a,d=read_model(f,kind); A.append(a); T.append(d)
        if (k+1)%150==0: print(f'   {k+1}/{len(files)}',flush=True)
    da=xr.DataArray(np.concatenate(A),dims=('time','cell'),
                    coords={'time':np.concatenate(T),'cell':sea.grid_id.values},name=kind)
    da.to_netcdf(fo,encoding={kind:{'zlib':True,'complevel':4}})
    print(f'  -> {os.path.basename(fo)} {da.shape}',flush=True)
    return da.load()

SIM={}
for m in MODELS:
    for kind in ('wave','wind'):
        for tag in ('his','ssp126','ssp585'):
            SIM[(m,kind,tag)]=build_model(m,kind,tag)

print('\n=== Check 1: wind/wave time axes ===')
for m in MODELS:
    for tag in ('his','ssp126','ssp585'):
        a=pd.to_datetime(SIM[(m,'wave',tag)].time.values)
        b=pd.to_datetime(SIM[(m,'wind',tag)].time.values)
        n=min(len(a),len(b))
        print(f'  {m:8s} {tag:7s} wave{len(a):6d} wind{len(b):6d} first {n} days match='
              f'{np.array_equal(a.values[:n],b.values[:n])}'
              + ('' if len(a)==len(b) else f'  ⚠wind missing {len(a)-len(b)} days'))


In [ ]:
def era5_wave_year(y, blk=15):
    ds=xr.open_dataset(os.path.join(E5_WV,f'era5_swh_{y}.nc'),chunks={'valid_time':240})
    da=ds['swh'].squeeze(drop=True); t=pd.to_datetime(ds['valid_time'].values)
    ndt=len(t)//24; A,T=[],[]
    for s in range(0,ndt,blk):
        n=min(blk,ndt-s); k=slice(s*24,(s+n)*24)
        x=da.isel(valid_time=k).values.astype(np.float32)
        dm=np.nanmax(x.reshape(n,24,361,720),axis=1); del x
        A.append(cell_05(dm[:,::-1,:])); T.append(t[k][::24].normalize()); del dm
    ds.close(); return np.concatenate(A), np.concatenate(T)

def era5_wind_year(y, blk=5):
    ds=xr.open_dataset(os.path.join(E5_WN,f'era5_wind10m_{y}.grib'),
                       engine='netcdf4',chunks={'valid_time':120})
    t=pd.to_datetime(ds['valid_time'].values); ndt=len(t)//24; A,T=[],[]
    for s in range(0,ndt,blk):
        n=min(blk,ndt-s); k=slice(s*24,(s+n)*24)
        u=ds['u10'].isel(valid_time=k).values.astype(np.float32)
        v=ds['v10'].isel(valid_time=k).values.astype(np.float32)
        spd=np.sqrt(u*u+v*v); del u,v                      # ★ combine at each hour first
        dm=np.nanmax(spd.reshape(n,24,721,1440),axis=1); del spd
        A.append(cell_025(dm[:,::-1,:])); T.append(t[k][::24].normalize()); del dm
        if (s//blk)%20==0: print(f'    {y} day {s}/{ndt}',flush=True)
    ds.close(); return np.concatenate(A), np.concatenate(T)

def from_m2_cache(kind, y0, y1):
    fr=[]
    for f in glob.glob(os.path.join(M2C,f'*__dmax_{kind}.parquet')):
        d=pd.read_parquet(f); d['date']=pd.to_datetime(d['date'])
        fr.append(d[(d.date.dt.year>=y0)&(d.date.dt.year<=y1)])
    d=pd.concat(fr,ignore_index=True).drop_duplicates(['grid_id','date'])
    p=(d.pivot(index='date',columns='grid_id',values='dmax')
         .sort_index().reindex(columns=sea.grid_id.values))
    return p.values.astype(np.float32), pd.DatetimeIndex(p.index)

def build_obs(kind):
    fo=os.path.join(OUT,f'obs_{kind}.nc')
    if os.path.exists(fo): print(f'[CACHE] obs_{kind}'); return xr.open_dataarray(fo).load()
    fn=era5_wave_year if kind=='wave' else era5_wind_year
    A,T=[],[]
    for y in range(REF[0],REF[1]+1):
        if M2_COVER[0]<=y<=M2_COVER[1]: continue
        a,t=fn(y); A.append(a); T.append(np.asarray(t)); print(f'  obs {kind} {y} computed',flush=True)
    y0c,y1c=max(REF[0],M2_COVER[0]), min(REF[1],M2_COVER[1])
    if y0c<=y1c:
        a,t=from_m2_cache(kind,y0c,y1c); A.append(a); T.append(np.asarray(t))
        print(f'  obs {kind} {y0c}-{y1c} taken from M2 cache {a.shape}',flush=True)
    arr=np.concatenate(A); tim=pd.DatetimeIndex(np.concatenate(T))
    o=np.argsort(tim.values); arr,tim=arr[o],tim[o]
    exp=pd.date_range(f'{REF[0]}-01-01',f'{REF[1]}-12-31',freq='D')
    assert len(tim)==len(exp) and (tim==exp).all(), f'{kind} time axis incomplete {len(tim)}/{len(exp)}'
    da=xr.DataArray(arr,dims=('time','cell'),
                    coords={'time':tim.values,'cell':sea.grid_id.values},name=kind)
    da.to_netcdf(fo,encoding={kind:{'zlib':True,'complevel':4}})
    print(f'  -> obs_{kind}.nc {da.shape}',flush=True); return da.load()

OBS={k:build_obs(k) for k in ('wave','wind')}

print('\n=== Check 2: obs magnitudes ===')
for k in ('wave','wind'):
    o=OBS[k].values
    print(f'  {k:5s} valid {np.isfinite(o).mean()*100:5.2f}%  median {np.nanmedian(o):5.2f}'
          f'  P99 {np.nanpercentile(o,99):5.2f}  max {np.nanmax(o):5.2f}')

# Check 3 (optional, ~2 min for wave): own computation vs M2 cache must match
CHECK_PATH = True
if CHECK_PATH:
    a1,t1=era5_wave_year(2012); a2,t2=from_m2_cache('wave',2012,2012)
    assert np.array_equal(np.asarray(t1),np.asarray(t2)), 'dates do not match'
    d=np.abs(a1-a2)
    print(f'\n  wave 2012 two paths: max diff {np.nanmax(d):.6f}  '
          f'agreement {(d<1e-4).mean()*100:.2f}%   (should be ≈100%)')


In [ ]:
QG=np.linspace(0,1,NQ)

def qgrid(a):
    q=np.nanquantile(a,QG,axis=0).astype(np.float32)
    return np.maximum.accumulate(q,axis=0)

def inv_cdf(q,tau):
    bad=~np.isfinite(tau); t=np.where(bad,0.,tau)
    p=np.clip(t,0,1)*(NQ-1); lo=np.floor(p).astype(np.int32); np.clip(lo,0,NQ-2,out=lo)
    fr=(p-lo).astype(np.float32)
    a=np.take_along_axis(q,lo,0); b=np.take_along_axis(q,lo+1,0)
    r=a+(b-a)*fr; r[bad]=np.nan; return r

def tau_of(x,q,BIG=1e4):
    N=x.shape[1]; off=np.arange(N,dtype=np.float64)*BIG
    ok=np.isfinite(q).all(0)
    qq=np.where(ok,q,np.linspace(0,1,NQ)[:,None]).astype(np.float64)
    flat=(qq+off).T.ravel()
    idx=np.searchsorted(flat,(np.nan_to_num(x,nan=-1e9)+off).ravel(),side='left')
    idx=idx.reshape(x.shape)-(np.arange(N)*NQ)
    hi=np.clip(idx,1,NQ-1).astype(np.int32); lo=hi-1
    a=np.take_along_axis(qq,lo,0); b=np.take_along_axis(qq,hi,0)
    den=b-a; fr=np.where(den>1e-9,(x-a)/np.where(den>1e-9,den,1.),0.)
    tau=np.clip((lo+np.clip(fr,0,1))/(NQ-1),0,1).astype(np.float32)
    tau[~np.isfinite(x)]=np.nan; tau[:,~ok]=np.nan
    return tau

def qdm_block(kind, model, tag, y0, y1):
    ov=OBS[kind].values; om=pd.to_datetime(OBS[kind].time.values).month.values
    h=SIM[(model,kind,'his')]; ht=pd.to_datetime(h.time.values)
    hk=(ht.year>=REF[0])&(ht.year<=REF[1]); hv=h.values[hk]; hm=ht.month.values[hk]
    p=SIM[(model,kind,tag)]; pt=pd.to_datetime(p.time.values)
    pk=(pt.year>=y0)&(pt.year<=y1); pv=p.values[pk]; pt=pt[pk]; pm=pt.month.values
    if len(pt)==0: return None,None
    out=np.full_like(pv,np.nan)
    for m in range(1,13):
        w=win(m); fk=np.isin(pm,w); ak=(pm==m)
        if fk.sum()<50 or not ak.any(): continue
        oq=qgrid(ov[np.isin(om,w)]); hq=qgrid(hv[np.isin(hm,w)]); pq=qgrid(pv[fk])
        x=pv[ak]; tau=tau_of(x,pq)
        hi=inv_cdf(hq,tau); oi=inv_cdf(oq,tau)
        with np.errstate(divide='ignore',invalid='ignore'):
            delta=x/np.where(hi>1e-3,hi,np.nan)
        out[ak]=np.clip(oi*delta,0,None).astype(np.float32)
    return out, pt

QDM={}
for model in MODELS:
    for kind in ('wave','wind'):
        for tag in ('his','ssp126','ssp585'):
            fo=os.path.join(OUT,f'qdm_{kind}_{model}_{tag}.nc')
            if os.path.exists(fo):
                QDM[(model,kind,tag)]=xr.open_dataarray(fo).load()
                print(f'[CACHE] {os.path.basename(fo)}'); continue
            A,T=[],[]
            for (y0,y1) in ([REF] if tag=='his' else BLOCKS):
                a,t=qdm_block(kind,model,tag,y0,y1)
                if a is not None: A.append(a); T.append(np.asarray(t))
            da=xr.DataArray(np.concatenate(A),dims=('time','cell'),
                            coords={'time':np.concatenate(T),'cell':sea.grid_id.values},name=kind)
            da.to_netcdf(fo,encoding={kind:{'zlib':True,'complevel':4}})
            QDM[(model,kind,tag)]=da.load()
            print(f'  -> {os.path.basename(fo)} {da.shape}',flush=True)

print('\n=== Check 4: QDM validity: reference-period N(x), d/yr ===')
ny=REF[1]-REF[0]+1
for kind,cuts in (('wave',HS_CUTS),('wind',WD_CUTS)):
    o=OBS[kind].values
    for c in cuts:
        s=f'  {kind:4s} N(≥{c:>5}) ERA5 {np.nanmean((o>=c).sum(0))/ny:7.2f}'
        for model in MODELS:
            q=QDM[(model,kind,'his')].values
            raw=SIM[(model,kind,'his')]; rt=pd.to_datetime(raw.time.values)
            rk=(rt.year>=REF[0])&(rt.year<=REF[1])
            s+=(f' │ {model:7s} corrected {np.nanmean((q>=c).sum(0))/ny:6.2f}'
                f' uncorrected {np.nanmean((raw.values[rk]>=c).sum(0))/ny:6.2f}')
        print(s)


In [ ]:
def counts_by_ym(arr,times,cuts,edges):
    ym=times.year.values*100+times.month.values
    u=np.unique(ym); idx=np.searchsorted(ym,u); ok=np.isfinite(arr)
    N={c:np.add.reduceat((ok&(arr>=c)).astype(np.float32),idx,0) for c in cuts}
    D={f'{edges[i]:g}_{edges[i+1]:g}':
       np.add.reduceat((ok&(arr>=edges[i])&(arr<edges[i+1])).astype(np.float32),idx,0)
       for i in range(len(edges)-1)}
    return N,D,np.add.reduceat(ok.astype(np.float32),idx,0),u

def region_table(model,kind,tag):
    cuts,edges=(HS_CUTS,HS_EDGES) if kind=='wave' else (WD_CUTS,WD_EDGES)
    da=QDM[(model,kind,tag)]; N,D,V,u=counts_by_ym(da.values,pd.to_datetime(da.time.values),cuts,edges)
    reg=sea.region.values; fr=[]
    for r in REGS:
        k=reg==r
        d=pd.DataFrame({'year':u//100,'cmonth':u%100,'region':r,'n_days':V[:,k].mean(1)})
        for c in cuts: d[f'N_{c:g}']=N[c][:,k].mean(1)
        for b,v in D.items(): d[f'D_{b}']=v[:,k].mean(1)
        fr.append(d)
    o=pd.concat(fr,ignore_index=True)
    o.insert(0,'kind',kind); o.insert(0,'tag',tag); o.insert(0,'model',model); return o

MAIN=pd.concat([region_table(m,k,t) for m in MODELS for k in ('wave','wind')
                for t in ('his','ssp126','ssp585')],ignore_index=True)
MAIN.to_csv(os.path.join(OUT,'main_absolute.csv'),index=False)

print('=== Check 5: bin closure ===')
for kind in ('wave','wind'):
    s=MAIN[MAIN.kind==kind]; dc=[c for c in s.columns if c.startswith('D_') and s[c].notna().any()]
    print(f'  {kind}: max error {np.abs(s[dc].sum(1)-s.n_days).max():.5f} (should be 0)')

VAL=[c for c in MAIN.columns if c.startswith(('N_','D_'))]+['n_days']
ANN=MAIN.drop(columns='cmonth').groupby(['model','kind','tag','region','year'],
                                        as_index=False)[VAL].sum()

def slice_change(kind,col,sl='2081-2100'):
    y0,y1=SLICES[sl]; rows=[]
    for m in MODELS:
        b=ANN[(ANN.model==m)&(ANN.kind==kind)&(ANN.tag=='his')&ANN.year.between(*REF)]
        for ssp in ('ssp126','ssp585'):
            f=ANN[(ANN.model==m)&(ANN.kind==kind)&(ANN.tag==ssp)&ANN.year.between(y0,y1)]
            for r in REGS:
                bv=b[b.region==r][col].mean(); ff=f[f.region==r]
                fv=ff[col].mean(); nyr=ff.year.nunique()
                rows.append(dict(model=m,scenario=ssp,slice=sl,region=r,metric=col,
                                 base=bv,fut=fv,delta=fv-bv,n_years=nyr,
                                 pct=np.nan if (not np.isfinite(bv) or bv<MIN_BASE)
                                     else (fv-bv)/bv*100))
    return pd.DataFrame(rows)

CHG=pd.concat([slice_change('wave',f'N_{c:g}',s) for c in HS_CUTS for s in SLICES]+
              [slice_change('wind',f'N_{c:g}',s) for c in WD_CUTS for s in SLICES],
              ignore_index=True)
CHG.to_csv(os.path.join(OUT,'change_by_slice.csv'),index=False)
print('\n=== 2081-2100 change (d/yr) ===')
print(CHG[CHG['slice']=='2081-2100']
      .pivot_table(index=['region','model'],columns=['metric','scenario'],values='delta')
      .round(2).to_string())
print('\nReference-period baseline (d/yr):')
print(CHG[(CHG['slice']=='2081-2100')&(CHG.scenario=='ssp585')]
      .pivot_table(index='region',columns=['metric','model'],values='base').round(2).to_string())


In [ ]:
PCTS={'P95':0.95,'P99':0.99}

def rel_thr(model,kind):
    da=SIM[(model,kind,'his')]; t=pd.to_datetime(da.time.values)
    k=(t.year>=REF[0])&(t.year<=REF[1]); d=da.values[k]; mon=t.month.values[k]
    subs=sea.subregion.values; out={p:np.full((12,NCELL),np.nan,np.float32) for p in PCTS}
    for m in range(1,13):
        a=d[np.isin(mon,win(m))]
        for s in np.unique(subs):
            kk=subs==s; x=a[:,kk].ravel(); x=x[np.isfinite(x)]
            if len(x)<500: continue
            for p,q in PCTS.items(): out[p][m-1,kk]=np.quantile(x,q)
    return out
RTHR={(m,k):rel_thr(m,k) for m in MODELS for k in ('wave','wind')}

def four_class(model,tag,ptag='P95'):
    dv=SIM[(model,'wave',tag)]; dw=SIM[(model,'wind',tag)]
    tv=pd.to_datetime(dv.time.values); tw=pd.to_datetime(dw.time.values)
    common=tv.intersection(tw)                              # ★ handle missing tail of EC-Earth3 ssp585 wind
    v=dv.sel(time=common.values).values; w=dw.sel(time=common.values).values
    mon=common.month.values
    ok=np.isfinite(v)&np.isfinite(w)
    ew=ok&(v>=RTHR[(model,'wave')][ptag][mon-1])
    ei=ok&(w>=RTHR[(model,'wind')][ptag][mon-1])
    return dict(wave_ext=ew,wind_ext=ei,wave_only=ew&~ei,wind_only=ei&~ew,
                compound=ew&ei,either=ew|ei,valid=ok), common

def ym_region(mask,times):
    ym=times.year.values*100+times.month.values
    u=np.unique(ym); idx=np.searchsorted(ym,u)
    cnt=np.add.reduceat(mask.astype(np.float32),idx,0); reg=sea.region.values
    o=pd.DataFrame({r:cnt[:,reg==r].mean(1) for r in REGS})
    o.insert(0,'cmonth',u%100); o.insert(0,'year',u//100); return o

SEC=[]
for model in MODELS:
    for tag in ('his','ssp126','ssp585'):
        c,t=four_class(model,tag)
        if tag=='his':
            a,b=int(c['wind_only'].sum()),int(c['wave_only'].sum())
            print(f'{model} self-check wind_only/wave_only={a/b:.4f} (should be ≈1)  '
                  f'sum of three classes==union {np.array_equal(c["wind_only"]+c["wave_only"]+c["compound"],c["either"])}')
        for h in ('wave_ext','wind_ext','compound','wave_only','wind_only','valid'):
            d=ym_region(c[h],t); d.insert(0,'hazard',h); d.insert(0,'tag',tag)
            d.insert(0,'model',model); SEC.append(d)
SEC=pd.concat(SEC,ignore_index=True)
SEC.to_csv(os.path.join(OUT,'secondary_relative.csv'),index=False)

LIFT=[]
for r in REGS:
    d=(SEC[SEC.hazard.isin(['wave_ext','wind_ext','compound','valid'])]
       .pivot_table(index=['model','tag','year','cmonth'],columns='hazard',values=r).reset_index())
    d['lift']=d.compound/(d.wave_ext*d.wind_ext/d.valid.replace(0,np.nan))
    d['region']=r; LIFT.append(d)
LIFT=pd.concat(LIFT,ignore_index=True); LIFT.to_csv(os.path.join(OUT,'coupling_lift.csv'),index=False)
print('\nReference-period wind-wave coupling lift (1 = independent):')
print(LIFT[(LIFT.tag=='his')&LIFT.year.between(*REF)]
      .groupby(['region','model']).lift.mean().round(2).unstack().to_string())


In [ ]:
ROLL=11
REG_LABEL={'NW_Europe':'Northwest Europe','NW_Atlantic':'Northwest Atlantic',
           'Med_Black':'Mediterranean & Black Sea','Asia':'East & Southeast Asia',
           'Baltic':'Baltic Sea','Indian_Ocean_N':'North Indian Ocean'}
REGS_ORD=['NW_Europe','NW_Atlantic','Med_Black','Asia','Baltic','Indian_Ocean_N']
C_HIS,SC='#3d4348',{'ssp126':'#2b7bba','ssp585':'#c8452b'}
DASH={m:d for m,d in zip(MODELS,['-',(0,(5,2))])}

def six_panel(getter, ylab, title):
    fig,axes=plt.subplots(2,3,figsize=(17.5,9.6),sharex=True,dpi=110)
    for ax,reg in zip(axes.ravel(),REGS_ORD):
        for m in MODELS:
            yb,vb=getter(m,'his',reg)
            if len(yb)==0: continue
            base=np.nanmean(vb[(yb>=REF[0])&(yb<=REF[1])])
            for tag,col in [('his',C_HIS),('ssp126',SC['ssp126']),('ssp585',SC['ssp585'])]:
                y,v=getter(m,tag,reg)
                if len(y)==0: continue
                s=pd.Series(v-base,index=y).sort_index()
                ax.plot(s.index,s.values,color=col,lw=.6,alpha=.25,zorder=2)
                ax.plot(s.index,s.rolling(ROLL,center=True,min_periods=ROLL//2+1).mean(),
                        ls=DASH[m],color=col,lw=2.3,zorder=4)
        ax.axhline(0,color='#4a4a4a',lw=1,zorder=1)
        ax.axvspan(REF[0],REF[1],color='#000',alpha=.05,lw=0,zorder=0)
        ax.set_title(REG_LABEL[reg],fontsize=13,pad=8); ax.set_xlim(REF[0],2100)
        ax.grid(alpha=.2,lw=.6)
        for sp in ('top','right'): ax.spines[sp].set_visible(False)
    for ax in axes[:,0]: ax.set_ylabel(ylab,fontsize=12)
    for ax in axes[-1,:]: ax.set_xlabel('Year',fontsize=12)
    hd=([Line2D([],[],color=C_HIS,lw=2.4,label='Historical')]+
        [Line2D([],[],color=SC[s],lw=2.4,label=SSP_LABEL[s]) for s in SC]+
        [Line2D([],[],color='#1a1a1a',ls=DASH[m],lw=1.9,label=MDL_LABEL[m]) for m in MODELS])
    fig.legend(handles=hd,loc='upper center',ncol=5,frameon=False,fontsize=12,
               bbox_to_anchor=(.5,1.005))
    fig.suptitle(title,y=1.05,fontsize=15); plt.tight_layout(); plt.show()

def g_main(kind,col):
    def f(model,tag,reg):
        d=ANN[(ANN.model==model)&(ANN.kind==kind)&(ANN.tag==tag)&(ANN.region==reg)].sort_values('year')
        return d.year.values, d[col].values
    return f
def g_sec(hazard):
    def f(model,tag,reg):
        d=(SEC[(SEC.model==model)&(SEC.tag==tag)&(SEC.hazard==hazard)]
           .groupby('year',as_index=False)[reg].sum())
        return d.year.values, d[reg].values
    return f
def g_lift():
    def f(model,tag,reg):
        d=(LIFT[(LIFT.model==model)&(LIFT.tag==tag)&(LIFT.region==reg)]
           .groupby('year',as_index=False).lift.mean())
        return d.year.values, d.lift.values
    return f

for c in HS_CUTS:
    six_panel(g_main('wave',f'N_{c:g}'),'Δ days yr$^{-1}$',
              f'Days with $H_s$ ≥ {c} m  —  change vs {REF[0]}–{REF[1]}  (bias-corrected)')
for c in WD_CUTS:
    six_panel(g_main('wind',f'N_{c:g}'),'Δ days yr$^{-1}$',
              f'Days with $U_{{10}}$ ≥ {c} m s$^{{-1}}$  —  change vs {REF[0]}–{REF[1]}  (bias-corrected)')
six_panel(g_sec('compound'),'Δ days yr$^{-1}$',
          'Compound wind–wave extreme days  —  change vs reference  (model-relative P95, no bias correction)')
six_panel(g_lift(),'Δ lift',
          'Wind–wave coupling strength (lift)  —  change vs reference  (1 = independent)')

# ── summary forest plot: 6 regions × 4 thresholds × 2 models × 2 scenarios in one figure ──
def forest(kind, cuts, unit, title, sl='2081-2100'):
    d=CHG[(CHG['slice']==sl)&(CHG.metric.isin([f'N_{c:g}' for c in cuts]))]
    MK={m:mk for m,mk in zip(MODELS,['o','s'])}
    CL=plt.cm.viridis(np.linspace(.15,.82,len(cuts)))
    fig,axes=plt.subplots(1,2,figsize=(14,7.6),sharey=True,dpi=115)
    for ax,ssp in zip(axes,('ssp126','ssp585')):
        for i,reg in enumerate(REGS_ORD):
            y0=len(REGS_ORD)-1-i
            if i%2==0: ax.axhspan(y0-.5,y0+.5,color='#f4f5f6',lw=0,zorder=0)
            for j,c in enumerate(cuts):
                off=(j-(len(cuts)-1)/2)*.17
                for m in MODELS:
                    r=d[(d.region==reg)&(d.metric==f'N_{c:g}')&(d.scenario==ssp)&(d.model==m)]
                    if r.empty or not np.isfinite(r.delta.iloc[0]): continue
                    ax.plot(r.delta.iloc[0],y0+off,MK[m],color=CL[j],ms=8,
                            mfc=CL[j] if m==list(MODELS)[0] else 'none',mew=1.6,zorder=3)
        ax.axvline(0,color='#333',lw=1.1)
        ax.set_yticks(range(len(REGS_ORD)))
        ax.set_yticklabels([REG_LABEL[r] for r in REGS_ORD][::-1],fontsize=11)
        ax.set_title(SSP_LABEL[ssp],fontsize=13); ax.set_xlabel(f'Δ days yr$^{{-1}}$',fontsize=11.5)
        ax.grid(axis='x',alpha=.25,lw=.6)
        for sp in ('top','right'): ax.spines[sp].set_visible(False)
    hd=([Line2D([],[],ls='',marker='o',color=CL[j],ms=8,label=f'≥ {c:g} {unit}')
         for j,c in enumerate(cuts)]+
        [Line2D([],[],ls='',marker=MK[m],color='#333',ms=8,
                mfc='#333' if m==list(MODELS)[0] else 'none',label=MDL_LABEL[m]) for m in MODELS])
    fig.legend(handles=hd,loc='upper center',ncol=len(cuts)+2,frameon=False,fontsize=11,
               bbox_to_anchor=(.5,1.01))
    fig.suptitle(f'{title}  —  {sl} vs {REF[0]}–{REF[1]}',y=1.07,fontsize=14.5)
    plt.tight_layout(); plt.show()

forest('wave',HS_CUTS,'m',   'Change in exceedance days, significant wave height')
forest('wind',WD_CUTS,'m/s', 'Change in exceedance days, 10-m wind speed')
